# Packet P-035 — Temporal Stability (Train-Old, Test-New)

**Decision question:** Does the model trained on earlier-published data predict later-published data equally well, or is there temporal drift?

**Method:** Sort devices by Ref_ID (publication-time proxy). Train on the first {50%, 60%, 70%, 80%, 90%} of devices, test on the remainder. Compare tau-b to random-split baseline (0.289).

**Fastest falsifier:** Single 70/30 temporal split. If tau-b on newest 30% < 0.15, temporal drift is present.

**Success:** Tau-b on newest 30% ≥ 0.20.
**Kill:** Tau-b on newest 30% < 0.10.

In [1]:
"""Cell 1 — Load data, check Ref_ID availability."""
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from scipy.stats import kendalltau
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('perovskite_stability_clean.csv')
FEATURES = [
    'Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
    'MA', 'FA', 'Cs',
    'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
    'Perovskite_thickness', 'Cell_area_measured',
    'JV_default_Voc', 'JV_default_Jsc', 'JV_default_FF'
]
TARGET = 'Stability_PCE_T80'
ET_PARAMS = dict(n_estimators=700, max_features='sqrt', min_samples_split=3,
                 min_samples_leaf=1, bootstrap=False, random_state=42)

X = df[FEATURES].fillna(0)
y = np.log1p(df[TARGET])

# Check for Ref_ID or similar temporal proxy
ref_cols = [c for c in df.columns if 'ref' in c.lower() or 'id' in c.lower() or 'doi' in c.lower()]
print(f"Potential temporal proxy columns: {ref_cols}")

# Use the DataFrame index as a proxy for temporal ordering
# (data is typically ordered by Ref_ID in the Perovskite Database)
if 'Ref_ID' in df.columns:
    df_sorted = df.sort_values('Ref_ID').reset_index(drop=False)
    print(f"Ref_ID range: {df['Ref_ID'].min()} to {df['Ref_ID'].max()}")
    print(f"Unique Ref_IDs: {df['Ref_ID'].nunique()}")
    temporal_col = 'Ref_ID'
else:
    # Use original index as temporal proxy
    df_sorted = df.reset_index(drop=False)
    temporal_col = 'index'
    print("No Ref_ID found — using original row index as temporal proxy")

print(f"Dataset: {len(df)} devices, sorted by {temporal_col}")

Potential temporal proxy columns: ['Ref_ID', 'ETL_deposition_synthesis_atmosphere_relative_humidity', 'HTL_deposition_synthesis_atmosphere_relative_humidity', 'Perovskite_deposition_thermal_annealing_relative_humidity', 'Perovskite_deposition_synthesis_atmosphere_relative_humidity']
Ref_ID range: 26 to 43752
Unique Ref_IDs: 1543
Dataset: 1543 devices, sorted by Ref_ID


In [2]:
"""Cell 2 — Temporal split evaluation."""

# Sort by temporal proxy and run 5 split points
split_fractions = [0.50, 0.60, 0.70, 0.80, 0.90]

if 'Ref_ID' in df.columns:
    sort_order = df['Ref_ID'].argsort().values
else:
    sort_order = np.arange(len(df))

X_sorted = X.iloc[sort_order]
y_sorted = y.iloc[sort_order]

results = []
print(f"{'Split':<10} {'Train':>7} {'Test':>7} {'Tau-b':>8} {'vs 0.289':>10}")
print("-" * 45)

for frac in split_fractions:
    n_train = int(len(df) * frac)
    n_test = len(df) - n_train
    
    X_tr = X_sorted.iloc[:n_train]
    y_tr = y_sorted.iloc[:n_train]
    X_te = X_sorted.iloc[n_train:]
    y_te = y_sorted.iloc[n_train:]
    
    model = ExtraTreesRegressor(**ET_PARAMS)
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    
    tau, p = kendalltau(y_te, preds)
    ratio = tau / 0.289 if 0.289 > 0 else 0
    
    results.append({
        'split': f"{int(frac*100)}/{int((1-frac)*100)}",
        'train_n': n_train, 'test_n': n_test,
        'tau_b': tau, 'ratio_to_baseline': ratio
    })
    
    marker = "✓" if tau >= 0.20 else ("~" if tau >= 0.10 else "✗")
    print(f"{int(frac*100)}/{int((1-frac)*100):<7} {n_train:>7} {n_test:>7} {tau:>8.4f} {ratio:>9.1%}  {marker}")

# Random baseline for comparison
from sklearn.model_selection import cross_val_score
print(f"\nRandom-split baseline: tau-b 0.289 (5-fold CV)")

# Key metric: 70/30 split
tau_70_30 = results[2]['tau_b']
print(f"\n── Fastest falsifier (70/30 split): tau-b = {tau_70_30:.4f} ──")
if tau_70_30 >= 0.20:
    print("  PASS: >= 0.20 threshold")
elif tau_70_30 >= 0.10:
    print("  WEAK: between 0.10 and 0.20")
else:
    print("  FAIL: below 0.10 — temporal drift detected")

Split        Train    Test    Tau-b   vs 0.289
---------------------------------------------


50/50          771     772   0.0965     33.4%  ✗


60/40          925     618   0.1092     37.8%  ~


70/30         1080     463   0.0927     32.1%  ✗


80/19         1234     309   0.1409     48.8%  ~


90/9          1388     155   0.0514     17.8%  ✗

Random-split baseline: tau-b 0.289 (5-fold CV)

── Fastest falsifier (70/30 split): tau-b = 0.0927 ──
  FAIL: below 0.10 — temporal drift detected


In [3]:
"""Cell 3 — Evaluate and save."""

# Success: tau-b on newest 30% >= 0.20
# Kill: tau-b on newest 30% < 0.10

# Check for degradation trend
taus = [r['tau_b'] for r in results]
is_degrading = all(taus[i] >= taus[i+1] for i in range(len(taus)-1))

print("── Temporal Degradation Analysis ──\n")
print(f"Monotonic degradation trend: {'YES' if is_degrading else 'NO'}")
print(f"Tau-b range: {min(taus):.4f} to {max(taus):.4f}")
print(f"70/30 split tau-b: {tau_70_30:.4f}")

if tau_70_30 >= 0.20:
    status = "Confirmed"
elif tau_70_30 >= 0.10:
    status = "Promising"
else:
    status = "Negative"

# Save
pd.DataFrame(results).to_csv('Packet_P035_Temporal_Stability.csv', index=False)
print(f"\nSaved: Packet_P035_Temporal_Stability.csv")

print(f"\n{'=' * 60}")
print(f"P-035 STATUS: {status}")
print(f"{'=' * 60}")
if status == "Confirmed":
    print("Model generalizes across publication time — no significant temporal drift.")
elif status == "Promising":
    print("Some temporal degradation but model retains weak predictive power.")
else:
    print("Significant temporal drift — model trained on old data fails on new.")
    print("Retraining cadence is important for deployment.")

── Temporal Degradation Analysis ──

Monotonic degradation trend: NO
Tau-b range: 0.0514 to 0.1409
70/30 split tau-b: 0.0927

Saved: Packet_P035_Temporal_Stability.csv

P-035 STATUS: Negative
Significant temporal drift — model trained on old data fails on new.
Retraining cadence is important for deployment.
